# IDEA-009: Regime Filter Testing

**Problem:** Our strategies are long-only but we haven't filtered for market regime.

**Questions:**
1. How did strategies perform during bull vs bear markets?
2. Which regime filter works best?
3. How much does a regime filter improve results?

**Regime Filter Options:**
- Price > 200 MA (simple trend)
- 50 MA > 200 MA (golden cross)
- MVRV > 1.0 (on-chain: market > realized cap)
- Price > Realized Price (on-chain: above aggregate cost basis)

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

print("Regime Filter Testing 🔍")

In [ ]:
# Load all data
DATA_DIR = Path("../data/daily")

price = pd.read_parquet(DATA_DIR / "price.parquet").rename(columns={"value": "price"}).set_index("time")
mvrv = pd.read_parquet(DATA_DIR / "mvrv.parquet").rename(columns={"value": "mvrv"}).set_index("time")
sopr = pd.read_parquet(DATA_DIR / "sopr.parquet").rename(columns={"value": "sopr"}).set_index("time")
sopr_sth = pd.read_parquet(DATA_DIR / "sopr_sth.parquet").rename(columns={"value": "sopr_sth"}).set_index("time")
realized_loss = pd.read_parquet(DATA_DIR / "realized_loss.parquet").rename(columns={"value": "realized_loss"}).set_index("time")

# Try to load realized price if available
try:
    realized_price = pd.read_parquet(DATA_DIR / "realized_price.parquet").rename(columns={"value": "realized_price"}).set_index("time")
    has_realized_price = True
except:
    has_realized_price = False
    print("Note: realized_price.parquet not found, will skip that filter")

df = price.join(mvrv, how='inner').join(sopr, how='inner').join(sopr_sth, how='inner').join(realized_loss, how='inner')
if has_realized_price:
    df = df.join(realized_price, how='inner')
df = df.sort_index()

# Create technical indicators
df['ma50'] = df['price'].rolling(50).mean()
df['ma200'] = df['price'].rolling(200).mean()

# Create RL z-score
df['rl_ma30'] = df['realized_loss'].rolling(30).mean()
df['rl_std30'] = df['realized_loss'].rolling(30).std()
df['rl_zscore'] = (df['realized_loss'] - df['rl_ma30']) / df['rl_std30']

df_test = df[df.index >= '2018-12-15'].copy()
df_test = df_test.dropna()

print(f"Data: {len(df_test)} rows")
print(f"Date range: {df_test.index.min().date()} to {df_test.index.max().date()}")

---
## 1. Define Market Regimes

In [ ]:
# Define regime filters
df_test['regime_ma200'] = df_test['price'] > df_test['ma200']  # Price > 200 MA
df_test['regime_golden'] = df_test['ma50'] > df_test['ma200']  # 50 MA > 200 MA (golden cross)
df_test['regime_mvrv'] = df_test['mvrv'] > 1.0  # MVRV > 1 (market > realized cap)

if has_realized_price:
    df_test['regime_rp'] = df_test['price'] > df_test['realized_price']  # Price > Realized Price

print("REGIME FILTER SUMMARY")
print("="*60)
print(f"{'Filter':<25} {'Bull Days':>12} {'Bear Days':>12} {'Bull %':>10}")
print("-"*60)

filters = ['regime_ma200', 'regime_golden', 'regime_mvrv']
if has_realized_price:
    filters.append('regime_rp')

for f in filters:
    bull_days = df_test[f].sum()
    bear_days = (~df_test[f]).sum()
    bull_pct = bull_days / len(df_test) * 100
    print(f"{f:<25} {bull_days:>12} {bear_days:>12} {bull_pct:>9.0f}%")

In [ ]:
# Visualize regimes
fig = make_subplots(rows=3, cols=1, shared_xaxes=True, row_heights=[0.5, 0.25, 0.25],
                    subplot_titles=['BTC Price with MAs', 'MVRV', 'Regime Indicators'])

# Price and MAs
fig.add_trace(go.Scatter(x=df_test.index, y=df_test['price'], name='Price'), row=1, col=1)
fig.add_trace(go.Scatter(x=df_test.index, y=df_test['ma50'], name='50 MA', line=dict(dash='dot')), row=1, col=1)
fig.add_trace(go.Scatter(x=df_test.index, y=df_test['ma200'], name='200 MA', line=dict(dash='dash')), row=1, col=1)

# MVRV
fig.add_trace(go.Scatter(x=df_test.index, y=df_test['mvrv'], name='MVRV'), row=2, col=1)
fig.add_hline(y=1.0, line_dash='dash', line_color='red', row=2, col=1)

# Regime indicators (as colored background)
fig.add_trace(go.Scatter(x=df_test.index, y=df_test['regime_ma200'].astype(int), 
                         name='Price>200MA', fill='tozeroy'), row=3, col=1)

fig.update_yaxes(type='log', row=1, col=1)
fig.update_layout(height=800, title_text='Market Regimes Over Test Period')
fig.show()

In [ ]:
# Identify distinct bear market periods
bear_ma200 = ~df_test['regime_ma200']
bear_ma200_shifted = bear_ma200.shift(1).fillna(False)
bear_starts = bear_ma200 & ~bear_ma200_shifted

print("\nBEAR MARKET PERIODS (Price < 200 MA)")
print("="*80)

# Find continuous bear periods
df_test['bear_period'] = (~df_test['regime_ma200']).astype(int)
df_test['bear_group'] = (df_test['bear_period'].diff() != 0).cumsum()

bear_periods = df_test[df_test['bear_period'] == 1].groupby('bear_group').agg({
    'price': ['first', 'min', 'last', 'count']
})
bear_periods.columns = ['start_price', 'min_price', 'end_price', 'days']

# Get dates
bear_dates = df_test[df_test['bear_period'] == 1].groupby('bear_group').apply(
    lambda x: (x.index.min(), x.index.max())
)

# Filter to significant periods (> 30 days)
significant_bears = bear_periods[bear_periods['days'] > 30]

print(f"\nSignificant bear periods (>30 days): {len(significant_bears)}")
print("-"*80)

for idx in significant_bears.index:
    if idx in bear_dates.index:
        start, end = bear_dates[idx]
        row = significant_bears.loc[idx]
        drawdown = (row['min_price'] / row['start_price'] - 1) * 100
        print(f"{start.date()} to {end.date()}: {int(row['days'])} days, "
              f"${row['start_price']:,.0f} → ${row['min_price']:,.0f} ({drawdown:.0f}%)")

---
## 2. Performance by Regime (No Filter)

In [ ]:
# Backtest functions
def sopr_rl_entry(df, rl_z_threshold=0.5):
    """STRAT-002 entry: SOPR + STH-SOPR + RL Z"""
    sopr_signal = (df['sopr'] < 1) & (df['sopr_sth'] < 1)
    rl_signal = df['rl_zscore'] > rl_z_threshold
    combined = sopr_signal & rl_signal
    entries = combined & ~combined.shift(1).fillna(False)
    return entries

def sopr_only_entry(df):
    """STRAT-001 entry: SOPR + STH-SOPR only"""
    sopr_signal = (df['sopr'] < 1) & (df['sopr_sth'] < 1)
    entries = sopr_signal & ~sopr_signal.shift(1).fillna(False)
    return entries

def backtest_mvrv_trailing(df, entries, mvrv_trigger=2.0, trailing_pct=0.25, stop_loss=0.20, max_hold_days=365):
    """Backtest with MVRV trailing exit."""
    trades = []
    entry_indices = entries[entries].index.tolist()
    close = df['price']
    
    i = 0
    while i < len(entry_indices):
        entry_date = entry_indices[i]
        entry_idx = df.index.get_loc(entry_date)
        entry_price = close.iloc[entry_idx]
        
        peak_price = entry_price
        trailing_active = False
        
        exit_date = None
        exit_price = None
        exit_reason = None
        
        for j in range(entry_idx + 1, len(df)):
            current_date = df.index[j]
            current_price = close.iloc[j]
            current_mvrv = df['mvrv'].iloc[j]
            days_held = j - entry_idx
            
            if current_price > peak_price:
                peak_price = current_price
            
            pnl = (current_price - entry_price) / entry_price
            
            if not trailing_active and current_mvrv >= mvrv_trigger:
                trailing_active = True
            
            if trailing_active:
                trail_stop = peak_price * (1 - trailing_pct)
                if current_price <= trail_stop:
                    exit_date = current_date
                    exit_price = trail_stop
                    exit_reason = 'mvrv_trail'
                    break
            
            if not trailing_active and stop_loss and pnl <= -stop_loss:
                exit_date = current_date
                exit_price = entry_price * (1 - stop_loss)
                exit_reason = 'stop_loss'
                break
            
            if days_held >= max_hold_days:
                exit_date = current_date
                exit_price = current_price
                exit_reason = 'max_hold'
                break
        
        if exit_date is None:
            exit_date = df.index[-1]
            exit_price = close.iloc[-1]
            exit_reason = 'end_of_data'
        
        pnl = (exit_price - entry_price) / entry_price
        
        # Determine regime at entry
        regime_bull = df.loc[entry_date, 'regime_ma200'] if 'regime_ma200' in df.columns else True
        
        trades.append({
            'entry_date': entry_date,
            'exit_date': exit_date,
            'entry_price': entry_price,
            'exit_price': exit_price,
            'pnl_pct': pnl,
            'days_held': (exit_date - entry_date).days,
            'exit_reason': exit_reason,
            'regime_bull': regime_bull
        })
        
        while i < len(entry_indices) and entry_indices[i] <= exit_date:
            i += 1
    
    return pd.DataFrame(trades)

In [ ]:
# Run STRAT-002 without any regime filter
entries_no_filter = sopr_rl_entry(df_test, 0.5)
trades_no_filter = backtest_mvrv_trailing(df_test, entries_no_filter)

print("STRAT-002 PERFORMANCE BY REGIME (NO FILTER)")
print("="*80)
print(f"\nTotal trades: {len(trades_no_filter)}")

if len(trades_no_filter) > 0:
    bull_trades = trades_no_filter[trades_no_filter['regime_bull']]
    bear_trades = trades_no_filter[~trades_no_filter['regime_bull']]
    
    print(f"\n{'Regime':<15} {'Trades':>10} {'Win Rate':>12} {'Avg PnL':>12} {'Total PnL':>12}")
    print("-"*65)
    
    if len(bull_trades) > 0:
        bull_wr = (bull_trades['pnl_pct'] > 0).mean() * 100
        bull_avg = bull_trades['pnl_pct'].mean() * 100
        bull_total = ((1 + bull_trades['pnl_pct']).prod() - 1) * 100
        print(f"{'BULL':15} {len(bull_trades):>10} {bull_wr:>11.0f}% {bull_avg:>+11.1f}% {bull_total:>+11.0f}%")
    else:
        print(f"{'BULL':15} {0:>10} {'-':>12} {'-':>12} {'-':>12}")
    
    if len(bear_trades) > 0:
        bear_wr = (bear_trades['pnl_pct'] > 0).mean() * 100
        bear_avg = bear_trades['pnl_pct'].mean() * 100
        bear_total = ((1 + bear_trades['pnl_pct']).prod() - 1) * 100
        print(f"{'BEAR':15} {len(bear_trades):>10} {bear_wr:>11.0f}% {bear_avg:>+11.1f}% {bear_total:>+11.0f}%")
    else:
        print(f"{'BEAR':15} {0:>10} {'-':>12} {'-':>12} {'-':>12}")
    
    print("-"*65)
    all_wr = (trades_no_filter['pnl_pct'] > 0).mean() * 100
    all_avg = trades_no_filter['pnl_pct'].mean() * 100
    all_total = ((1 + trades_no_filter['pnl_pct']).prod() - 1) * 100
    print(f"{'ALL':15} {len(trades_no_filter):>10} {all_wr:>11.0f}% {all_avg:>+11.1f}% {all_total:>+11.0f}%")

In [ ]:
# Show individual trades
print("\nINDIVIDUAL TRADES")
print("="*120)
print(f"{'Entry Date':<12} {'Exit Date':<12} {'Entry $':>10} {'Exit $':>10} {'PnL':>8} {'Days':>6} {'Exit Reason':<12} {'Regime':<6}")
print("-"*120)

for _, trade in trades_no_filter.iterrows():
    regime = "BULL" if trade['regime_bull'] else "BEAR"
    print(f"{str(trade['entry_date'].date()):<12} {str(trade['exit_date'].date()):<12} "
          f"{trade['entry_price']:>10,.0f} {trade['exit_price']:>10,.0f} "
          f"{trade['pnl_pct']*100:>+7.1f}% {trade['days_held']:>6} {trade['exit_reason']:<12} {regime:<6}")

---
## 3. Test Regime Filters

In [ ]:
def walk_forward_with_regime(df, entry_func, regime_filter=None, mvrv_trigger=2.0, trailing_pct=0.25):
    """Walk-forward validation with optional regime filter."""
    results = []
    close = df['price']
    
    train_days = 365
    test_days = 90
    step_days = 90
    
    total_days = len(df)
    n_folds = (total_days - train_days) // step_days
    total_trades = 0
    
    for fold in range(n_folds):
        test_start = train_days + fold * step_days
        test_end = min(test_start + test_days, total_days)
        
        test_df = df.iloc[test_start:test_end].copy()
        test_close = close.iloc[test_start:test_end]
        
        # Get base entries
        entries = entry_func(test_df)
        
        # Apply regime filter if specified
        if regime_filter is not None:
            regime_ok = test_df[regime_filter]
            entries = entries & regime_ok
        
        trades = backtest_mvrv_trailing(test_df, entries, mvrv_trigger, trailing_pct)
        total_trades += len(trades)
        
        strat_return = (1 + trades['pnl_pct']).prod() - 1 if len(trades) > 0 else 0
        hold_return = (test_close.iloc[-1] / test_close.iloc[0]) - 1
        
        results.append({
            'fold': fold,
            'strat_return': strat_return,
            'hold_return': hold_return,
            'beat_hold': strat_return > hold_return,
            'n_trades': len(trades)
        })
    
    wf_df = pd.DataFrame(results)
    beat_rate = wf_df['beat_hold'].mean()
    avg_excess = (wf_df['strat_return'] - wf_df['hold_return']).mean()
    
    return beat_rate, avg_excess, total_trades, wf_df

In [ ]:
# Test all regime filters
print("REGIME FILTER COMPARISON")
print("="*100)
print(f"{'Filter':<25} {'Beat Rate':>12} {'Avg Excess':>12} {'Trades':>10} {'vs No Filter':>15}")
print("-"*100)

# No filter baseline
beat_no_filter, excess_no_filter, trades_no_filter_count, _ = walk_forward_with_regime(
    df_test, lambda df: sopr_rl_entry(df, 0.5), regime_filter=None
)
print(f"{'No Filter':<25} {beat_no_filter*100:>11.0f}% {excess_no_filter*100:>+11.1f}% {trades_no_filter_count:>10} {'-':>15}")

regime_results = [{
    'filter': 'No Filter',
    'beat_rate': beat_no_filter,
    'avg_excess': excess_no_filter,
    'trades': trades_no_filter_count
}]

# Test each filter
filters_to_test = [
    ('regime_ma200', 'Price > 200 MA'),
    ('regime_golden', '50 MA > 200 MA'),
    ('regime_mvrv', 'MVRV > 1.0'),
]

if has_realized_price:
    filters_to_test.append(('regime_rp', 'Price > Realized Price'))

for filter_col, filter_name in filters_to_test:
    beat_rate, avg_excess, n_trades, _ = walk_forward_with_regime(
        df_test, lambda df: sopr_rl_entry(df, 0.5), regime_filter=filter_col
    )
    
    diff = (beat_rate - beat_no_filter) * 100
    diff_str = f"{diff:+.0f}%" if diff != 0 else "0%"
    
    print(f"{filter_name:<25} {beat_rate*100:>11.0f}% {avg_excess*100:>+11.1f}% {n_trades:>10} {diff_str:>15}")
    
    regime_results.append({
        'filter': filter_name,
        'beat_rate': beat_rate,
        'avg_excess': avg_excess,
        'trades': n_trades
    })

regime_df = pd.DataFrame(regime_results)

In [ ]:
# Visualize results
fig = go.Figure()

colors = ['gray'] + ['green' if r['beat_rate'] > beat_no_filter else 'red' for r in regime_results[1:]]

fig.add_trace(go.Bar(
    x=[r['filter'] for r in regime_results],
    y=[r['beat_rate'] * 100 for r in regime_results],
    marker_color=colors,
    text=[f"{r['beat_rate']*100:.0f}%" for r in regime_results],
    textposition='outside'
))

fig.add_hline(y=beat_no_filter*100, line_dash='dash', line_color='gray', 
              annotation_text=f'No Filter: {beat_no_filter*100:.0f}%')
fig.add_hline(y=62, line_dash='dot', line_color='orange', 
              annotation_text='STRAT-001: 62%')

fig.update_layout(
    title='Beat Rate by Regime Filter',
    yaxis_title='Beat Rate %',
    height=500
)
fig.show()

---
## 4. Deep Dive: Best Filter

In [ ]:
# Find best filter
best_filter = max(regime_results, key=lambda x: x['beat_rate'])
print(f"\nBEST REGIME FILTER: {best_filter['filter']}")
print(f"Beat Rate: {best_filter['beat_rate']*100:.0f}%")
print(f"Avg Excess: {best_filter['avg_excess']*100:+.1f}%")
print(f"Trades: {best_filter['trades']}")

In [ ]:
# Compare trades with and without best filter
if best_filter['filter'] != 'No Filter':
    # Find the filter column
    filter_col = None
    for col, name in filters_to_test:
        if name == best_filter['filter']:
            filter_col = col
            break
    
    if filter_col:
        # Get trades with filter
        entries_filtered = sopr_rl_entry(df_test, 0.5) & df_test[filter_col]
        trades_filtered = backtest_mvrv_trailing(df_test, entries_filtered)
        
        # Get trades without filter (already have this)
        entries_no_filter = sopr_rl_entry(df_test, 0.5)
        trades_unfiltered = backtest_mvrv_trailing(df_test, entries_no_filter)
        
        print(f"\nTRADE COMPARISON: {best_filter['filter']}")
        print("="*80)
        print(f"{'Metric':<25} {'No Filter':>20} {'With Filter':>20}")
        print("-"*80)
        print(f"{'Total Trades':<25} {len(trades_unfiltered):>20} {len(trades_filtered):>20}")
        
        if len(trades_unfiltered) > 0 and len(trades_filtered) > 0:
            print(f"{'Win Rate':<25} {(trades_unfiltered['pnl_pct']>0).mean()*100:>19.0f}% {(trades_filtered['pnl_pct']>0).mean()*100:>19.0f}%")
            print(f"{'Avg PnL':<25} {trades_unfiltered['pnl_pct'].mean()*100:>+19.1f}% {trades_filtered['pnl_pct'].mean()*100:>+19.1f}%")
            print(f"{'Total Return':<25} {((1+trades_unfiltered['pnl_pct']).prod()-1)*100:>+19.0f}% {((1+trades_filtered['pnl_pct']).prod()-1)*100:>+19.0f}%")
            
            # Count by exit reason
            print(f"\nExit Reasons:")
            for reason in ['mvrv_trail', 'stop_loss', 'max_hold', 'end_of_data']:
                nf_count = (trades_unfiltered['exit_reason'] == reason).sum()
                f_count = (trades_filtered['exit_reason'] == reason).sum()
                print(f"  {reason:<20} {nf_count:>10} {f_count:>20}")

In [ ]:
# Show which trades were filtered out
if best_filter['filter'] != 'No Filter' and filter_col:
    print(f"\nTRADES FILTERED OUT by {best_filter['filter']}")
    print("="*100)
    
    # Find entries that would have been filtered
    entries_all = sopr_rl_entry(df_test, 0.5)
    regime_ok = df_test[filter_col]
    entries_filtered_out = entries_all & ~regime_ok
    
    filtered_dates = entries_filtered_out[entries_filtered_out].index.tolist()
    
    print(f"\nFiltered out {len(filtered_dates)} entry signals:")
    for date in filtered_dates:
        price = df_test.loc[date, 'price']
        mvrv = df_test.loc[date, 'mvrv']
        print(f"  {date.date()}: Price=${price:,.0f}, MVRV={mvrv:.2f}")

---
## 5. Test STRAT-001 with Regime Filter

In [ ]:
# Also test STRAT-001 (SOPR only) with regime filters
print("\nSTRAT-001 (SOPR ONLY) WITH REGIME FILTERS")
print("="*100)
print(f"{'Filter':<25} {'Beat Rate':>12} {'Avg Excess':>12} {'Trades':>10}")
print("-"*100)

# No filter baseline for STRAT-001
beat_s1_no_filter, excess_s1_no_filter, trades_s1_no_filter, _ = walk_forward_with_regime(
    df_test, sopr_only_entry, regime_filter=None, mvrv_trigger=2.25, trailing_pct=0.20
)
print(f"{'No Filter':<25} {beat_s1_no_filter*100:>11.0f}% {excess_s1_no_filter*100:>+11.1f}% {trades_s1_no_filter:>10}")

strat1_results = [{
    'filter': 'No Filter',
    'beat_rate': beat_s1_no_filter,
    'avg_excess': excess_s1_no_filter,
    'trades': trades_s1_no_filter
}]

for filter_col, filter_name in filters_to_test:
    beat_rate, avg_excess, n_trades, _ = walk_forward_with_regime(
        df_test, sopr_only_entry, regime_filter=filter_col, mvrv_trigger=2.25, trailing_pct=0.20
    )
    print(f"{filter_name:<25} {beat_rate*100:>11.0f}% {avg_excess*100:>+11.1f}% {n_trades:>10}")
    
    strat1_results.append({
        'filter': filter_name,
        'beat_rate': beat_rate,
        'avg_excess': avg_excess,
        'trades': n_trades
    })

---
## 6. Summary

In [ ]:
print("\n" + "="*80)
print("REGIME FILTER ANALYSIS SUMMARY")
print("="*80)

# Best filter for STRAT-002
best_s2 = max(regime_results, key=lambda x: x['beat_rate'])
print(f"\n📊 STRAT-002 (SOPR + RL)")
print(f"   Without filter: {beat_no_filter*100:.0f}% beat rate")
print(f"   Best filter: {best_s2['filter']}")
print(f"   With filter: {best_s2['beat_rate']*100:.0f}% beat rate")
print(f"   Improvement: {(best_s2['beat_rate'] - beat_no_filter)*100:+.0f}%")

# Best filter for STRAT-001
best_s1 = max(strat1_results, key=lambda x: x['beat_rate'])
print(f"\n📊 STRAT-001 (SOPR only)")
print(f"   Without filter: {beat_s1_no_filter*100:.0f}% beat rate")
print(f"   Best filter: {best_s1['filter']}")
print(f"   With filter: {best_s1['beat_rate']*100:.0f}% beat rate")
print(f"   Improvement: {(best_s1['beat_rate'] - beat_s1_no_filter)*100:+.0f}%")

# Verdict
print(f"\n🎯 VERDICT:")
if best_s2['beat_rate'] > beat_no_filter:
    print(f"   ✅ Regime filter IMPROVES STRAT-002!")
    print(f"   → Add '{best_s2['filter']}' to entry rules")
elif best_s2['beat_rate'] == beat_no_filter:
    print(f"   ⚠️ Regime filter doesn't change STRAT-002 beat rate")
    print(f"   → May still reduce drawdowns, needs more analysis")
else:
    print(f"   ❌ Regime filter HURTS STRAT-002")
    print(f"   → Contrarian signal may need bear market entries")

print("\n" + "="*80)

In [ ]:
# Save results
import json

def to_native(obj):
    if isinstance(obj, dict):
        return {k: to_native(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [to_native(v) for v in obj]
    elif hasattr(obj, 'item'):
        return obj.item()
    return obj

results = {
    'strat2_results': to_native(regime_results),
    'strat1_results': to_native(strat1_results),
    'best_filter_strat2': best_s2['filter'],
    'best_filter_strat1': best_s1['filter'],
    'baseline_strat2': beat_no_filter,
    'baseline_strat1': beat_s1_no_filter
}

with open('../data/regime_filter_results.json', 'w') as f:
    json.dump(results, f, indent=2)

print("Saved to ../data/regime_filter_results.json")